In [4]:
!pip install sentencepiece psutil

In [5]:
import psutil # for ram
import torch
import os
import random
import numpy as np
import torch.nn as nn
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [6]:
import sentencepiece as spm

# Train the model
spm.SentencePieceTrainer.train(
    input='training-data-expanded_3.txt',       # Your training data
    model_prefix='misovic',    # Output model name
    vocab_size=10000,          # Desired vocabulary size
    model_type='unigram',     # Algorithm: 'unigram' or 'bpe'
    character_coverage=1.0    # 1.0 for English, 0.9995 for CJK
)

sp = spm.SentencePieceProcessor(model_file='misovic.model')

with open("training-data-expanded_3.txt", "r", encoding="utf-8") as f:
    data = f.read()
token_ids = sp.encode(data, out_type=int) # convert to manual char level tokenization - token_ids, where data is the training corpus

In [7]:
#INPUTS, tokenization and embedding
vocab_size = 10000
embed_dim = 256
batch_size = 20
block_size = 64 # how big is the context window for X and Y, where Y is X shifted to the right, aka the next element after X
num_heads = 4
num_layers = 6 # how many blocks of attention/ff

In [8]:
#network architechture - attention + ff

class TransformerBlock(nn.Module):
    def __init__(self, embed_dim, num_heads, ff_hidden_dim=None):
        super().__init__()
        if ff_hidden_dim is None:
            ff_hidden_dim = embed_dim * 4
        #prevent looking into the future
        seq_len = block_size
        m = torch.full((seq_len, seq_len), float('-inf'))
        self.register_buffer('mask', torch.triu(m, diagonal=1))
        self.attn = nn.MultiheadAttention(embed_dim, num_heads)
        self.ln1 = nn.LayerNorm(embed_dim)
        self.ff = nn.Sequential(
            nn.Linear(embed_dim, ff_hidden_dim), # expand
            nn.ReLU(),
            nn.Linear(ff_hidden_dim, embed_dim) # contract
        )
        self.ln2 = nn.LayerNorm(embed_dim)
        self.dropout = nn.Dropout(0.1)

    def forward(self, x): #return logits of the architechture
        x = self.dropout(x) # before attn
        seq_len = x.size(0)
        attn_out, _ = self.attn(x, x, x, attn_mask = self.mask[:seq_len, :seq_len]) # object call → MultiheadAttention.forward
        #You want each token in each sequence to attend over all tokens in that same sequence, so:
        #query = key = value = x
        x = x + attn_out # residual connection
        x = self.ln1(x) # object call → LayerNorm.forward
        x = self.dropout(x) # before ff
        ff_out = self.ff(x) # object call → Sequential.forward
        x = x + ff_out # residual connection
        x = self.ln2(x)
        return x



In [9]:
#using the architechture to construct the lm

class lm(nn.Module):
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim) # return the token embed_dim sized vector
        self.pos_emb = nn.Embedding(block_size, embed_dim)
        self.blocks = nn.ModuleList([
            TransformerBlock(embed_dim, num_heads) for _ in range(num_layers)
        ])
        self.lm_head = nn.Linear(embed_dim, vocab_size) # return logits for the whole vocab

    def forward(self, batch):
        embedded = self.embedding(batch) # from the embedding matrix, return a vector for each batch element(its id)
        seq_len = batch.size(1) #block_size
        positions = torch.arange(seq_len, device=batch.device)   # tensor([0,1,...,seq_len-1]) za svaki token dobija se indeks da bi svaki token znao gde je
        pos = self.pos_emb(positions)                            # look those rows up
        x = embedded + pos #shape - 20, 64, 256 - batch_size, block_size, emb_dim
        emb_t = x.transpose(0,1) # transpose it because the block(transformer) expects 64, 20, 256
        for block in self.blocks:  # Passes the WHOLE batch through EACH layer, once per layer
            emb_t = block(emb_t) # calls forward for emb_t
        emb_t = emb_t.transpose(0,1)
        logits = self.lm_head(emb_t)

        return logits

model = lm(vocab_size, embed_dim)
model = model.to(device)

In [10]:
#load the previous model
if os.path.exists("model_checkpoint.pt"):
    model.load_state_dict(torch.load("model_checkpoint.pt", map_location=device))
    print("Resumed from checkpoint.")

In [ ]:
#training the model - making the dataset class

class LanguageModelDataset(torch.utils.data.Dataset):
    def __init__(self, token_ids, block_size):
        self.token_ids = token_ids
        self.block_size = block_size

    def __len__(self):
        return len(self.token_ids) - self.block_size

    def __getitem__(self, idx):
        x = torch.tensor(self.token_ids[idx : idx + self.block_size], dtype=torch.long)
        y = torch.tensor(self.token_ids[idx + 1 : idx + self.block_size + 1], dtype=torch.long)
        return x, y

from torch.utils.data import DataLoader

N = len(token_ids)
chunk=10000
chunks = [token_ids[i:i+chunk] for i in range(0,N,chunk)]
random.seed(42)
random.shuffle(chunks)
token_ids = [t for ch in chunks for t in ch]
train_ids = token_ids[:int(0.8 * N)]
val_ids   = token_ids[int(0.8 * N):int(0.9 * N)]
test_ids  = token_ids[int(0.9 * N):]

#training data is for train_ids
dataset = LanguageModelDataset(train_ids, block_size)
dataset_val = LanguageModelDataset(val_ids, block_size)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle=True, drop_last=True)
val_dataloader = DataLoader(dataset_val, batch_size = batch_size, shuffle=True, drop_last=True)

import torch.optim as optim

# Adam optimizer & epoch count:
optimizer = optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-4)
num_epochs = 2


loss_fn = nn.CrossEntropyLoss()
try:
    for epoch in range(num_epochs):
        c = 0
        total_loss = 0
        if epoch >= 1:
            for param_group in optimizer.param_groups:
                param_group['lr'] *= 0.33
            print("Learning rate lowered!")
        for batch_X, batch_Y in dataloader:
            #if c >= 25000: break

            #move from the cpu to gpu
            batch_X = batch_X.to(device)
            batch_Y = batch_Y.to(device)
            #forward pass
            logits = model(batch_X) # should be batch_size, block_size, vocab_size
            #print(logits.shape)
            # compute loss
            loss = loss_fn(logits.view(-1, vocab_size), batch_Y.view(-1))
            optimizer.zero_grad()  # clear old gradients
            loss.backward()        # backprop / compute gradients
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()       # update weights
            total_loss += loss.item()
            c+=1
            if c % 100 == 0 and psutil.virtual_memory().percent > 90:
                print("RAM is gettin filled up too much, stopping and saving...")
                torch.save(model.state_dict(), "model_checkpoint.pt")
                raise KeyboardInterrupt
            if c % 500 == 0:
                print(f"Iteration: {c}, loss: {loss.item():.4f}")
            if c % 5000 == 0:
                model.eval()
                with torch.no_grad():
                    vX, vY = next(iter(val_dataloader))
                    v_logits = model(vX.to(device))
                    v_loss = loss_fn(v_logits.view(-1, vocab_size), vY.to(device).view(-1))
                model.train()
                print(f"iter {c}: train {loss.item():.3f} | val {v_loss.item():.3f}")
        torch.save(model.state_dict(), "model_checkpoint.pt")
except KeyboardInterrupt:
    print("Training interrupted. Saving model...")
    torch.save(model.state_dict(), "model_checkpoint.pt")
    print("Model saved.")

Iteration: 500, loss: 6.2326
Iteration: 1000, loss: 6.0587
Iteration: 1500, loss: 5.8204
Iteration: 2000, loss: 6.0167
Iteration: 2500, loss: 5.8383
Iteration: 3000, loss: 5.7314
Iteration: 3500, loss: 5.4894
Iteration: 4000, loss: 5.6180
Iteration: 4500, loss: 5.5760
Iteration: 5000, loss: 5.7032
iter 5000: train 5.703 | val 5.376
Iteration: 5500, loss: 5.6034
Iteration: 6000, loss: 5.3047
Iteration: 6500, loss: 5.4256
Iteration: 7000, loss: 5.2266
Iteration: 7500, loss: 5.7428
Iteration: 8000, loss: 5.1800
Iteration: 8500, loss: 5.4171
Iteration: 9000, loss: 5.2372
Iteration: 9500, loss: 5.4439
Iteration: 10000, loss: 5.2824
iter 10000: train 5.282 | val 5.002
Iteration: 10500, loss: 5.2551
Iteration: 11000, loss: 5.1229
Iteration: 11500, loss: 5.4835
Iteration: 12000, loss: 5.3409
Iteration: 12500, loss: 5.4617
Iteration: 13000, loss: 5.5060
Iteration: 13500, loss: 5.2353
Iteration: 14000, loss: 5.3694
Iteration: 14500, loss: 5.2368
Iteration: 15000, loss: 5.4243
iter 15000: train 5

In [ ]:
#Validation to prevent overfitting:
model.eval()  # disables dropout/other randomness
total_val_loss = 0
loss_fn = nn.CrossEntropyLoss()
with torch.no_grad():
    for batch_X, batch_Y in val_dataloader:
        #move from the cpu to gpu
        batch_X = batch_X.to(device)
        batch_Y = batch_Y.to(device)
        #print("logits:", logits.shape)  # e.g., [32, 32, 1000]
        #forward pass
        logits = model(batch_X)
        #print("batch_y:", batch_Y.shape)  # e.g., [32, 32]
        loss = loss_fn(logits.view(-1, vocab_size), batch_Y.view(-1))
        total_val_loss += loss.item()
print(f"Epoch {epoch}: train_loss={total_loss/len(dataloader):.2f}, val_loss={total_val_loss/len(val_dataloader):.2f}")
#model.train()  # back to training mode

In [ ]:
#outputing the next token
prompt = input("Enter your prompt: ")
#logits = model(encoded_prompt)   →   (1, T, 10000)
#lm_head = (10000, 256)
def generate(prompt, max_tokens, temperature=0.7):
    model.eval() # no dropout anymore
    ids = sp.encode(prompt, out_type=int)
    encoded_prompt = torch.tensor([ids], dtype=torch.long, device=device) # [] to make it 2D
    with torch.no_grad():
        for _ in range(max_tokens):
            context = encoded_prompt[:, -block_size:] #model limitted to block_size, take the last block_size amount of tokens
            logits = model(context) #id's of the prompt, this gives us the next token
            last = logits[:, -1, :] # token id is the index of the matrix itself
            #[dim0:dim1:dim2], dim0 - batch, dim1 - token id out of them all, dim2- vocab
            probs = torch.softmax(last / temperature, dim=-1) #-1, ditch the other dimension, last is shape(1,10000) no T since we -1'd it in logits
            next_token = torch.multinomial(probs, num_samples=1) #multinomial picks 40% chance 40% of the time

            encoded_prompt = torch.cat([encoded_prompt, next_token], dim=1)
        decoded_prompt = sp.decode(encoded_prompt[0].tolist()) #back to text
    return decoded_prompt

generate(prompt, 200) #run the model